# 03 — Cross-country robustness

Identical specifications estimated separately for each country, before any
pooled model is considered.

The order is deliberate. A pooled coefficient imposes homogeneous transmission
dynamics; where countries genuinely differ, it is a weighted average of different
processes rather than a common parameter. Looking at the spread first makes that
visible instead of hiding it in one number.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from wage_transmission.config import load_models_config
from wage_transmission.cross_country import (
    estimate_country_robustness,
    summarise_country_robustness,
)
from wage_transmission.models.dynamic_panel import estimate_dynamic_panel

PROCESSED = Path("../data/processed/panel.csv")
ILLUSTRATIVE = not PROCESSED.exists()

if not ILLUSTRATIVE:
    panel = pd.read_csv(PROCESSED)
else:
    # No processed multi-country panel in this checkout. The frame below exists
    # only to exercise the interface: it is SIMULATED, and nothing estimated
    # from it is evidence about any real country.
    rng = np.random.default_rng(20260824)
    years = np.arange(1995, 2025)
    frames = []
    for code, beta in {"AAA": 0.4, "BBB": 0.7, "CCC": 0.9, "DDD": 1.1}.items():
        growth = rng.normal(0.017, 0.017, len(years))
        wage_growth = 0.001 + beta * growth + rng.normal(0, 0.007, len(years))
        frames.append(
            pd.DataFrame(
                {
                    "country": code,
                    "year": years,
                    "real_wage": 20000 * np.exp(np.cumsum(wage_growth)),
                    "productivity": 30 * np.exp(np.cumsum(growth)),
                }
            )
        )
    panel = pd.concat(frames, ignore_index=True)

config = load_models_config(Path("../config/models.yml"))
print("SIMULATED DATA — NOT EVIDENCE" if ILLUSTRATIVE else f"Source: {PROCESSED}")
print(f"Countries: {sorted(panel['country'].unique())}")

SIMULATED DATA — NOT EVIDENCE
Countries: ['AAA', 'BBB', 'CCC', 'DDD']


## Country-specific estimates

In [2]:
estimates = estimate_country_robustness(panel, config=config)
columns = [
    "country",
    "nobs",
    "distributed_lag_cumulative",
    "distributed_lag_cumulative_se",
    "cointegration_5pct",
]
estimates.loc[:, columns].sort_values("distributed_lag_cumulative").round(3)

,country,nobs,distributed_lag_cumulative,distributed_lag_cumulative_se,cointegration_5pct
1,BBB,30,0.693,0.210,True
0,AAA,30,0.771,0.188,False
2,CCC,30,0.894,0.178,False
3,DDD,30,1.456,0.165,False


## How different are they?

`I²` is the share of the observed variation in country estimates that exceeds what
sampling error alone would produce. A high value means a pooled number is
describing heterogeneous processes.

In [3]:
summary = summarise_country_robustness(estimates)
print(f"Countries              : {summary.n_countries}")
print(f"Median transmission    : {summary.median_cumulative_transmission:.3f}")
print(
    f"Interquartile range    : {summary.q25_cumulative_transmission:.3f}"
    f" to {summary.q75_cumulative_transmission:.3f}"
)
print(
    f"Random-effects estimate: {summary.random_effect_estimate:.3f}"
    f" (se {summary.random_effect_std_error:.3f})"
)
print(f"I-squared              : {summary.i_squared_percent:.1f}%")
print(f"Verdict                : {summary.interpretation}")

Countries              : 4
Median transmission    : 0.832
Interquartile range    : 0.751 to 1.034
Random-effects estimate: 0.965 (se 0.181)
I-squared              : 74.1%
Verdict                : moderate_cross_country_heterogeneity


## Only now: the pooled dynamic panel

A pooled estimate is only comparable with the country estimates above if it
targets the same quantity. The country models report a **cumulative multiplier**,
so the panel uses the same dynamic structure and reports
$\Theta = (\sum_j \beta_j)/(1-\gamma)$ rather than a contemporaneous slope.

Earlier releases of this notebook showed a static pooled regression beside the
cumulative country estimates. That comparison was wrong: the two are different
objects on different samples, and the static version is no longer reported.

A lagged dependent variable beside fixed effects biases least squares downward by
order $1/T$, so the estimate is bias-corrected; the uncorrected value is printed
beside it to show the size of the correction. **The correction addresses dynamic
fixed-effects bias only.** It does nothing about contemporaneous endogeneity
between productivity and wages.

In [4]:
# The published figures use 4,999 replications and take minutes per specification.
# This notebook uses far fewer so it stays runnable; the release artefact under
# results/vintages/<vintage>/ carries the numbers the paper reports.
panel_result = estimate_dynamic_panel(
    panel,
    fixed_effects="country_and_year",
    replications=299,
    bias_correction_draws=100,
)
low, high = panel_result.corrected_multiplier_ci
print(
    f"Observations      : {panel_result.nobs} ({panel_result.n_countries} countries,"
    f" {panel_result.n_effective_years} years)"
)
print(
    f"Uncorrected Theta : {panel_result.lsdv_multiplier:.3f} (gamma {panel_result.lsdv_persistence:.3f})"
)
print(
    f"Corrected Theta   : {panel_result.corrected_multiplier:.3f} (gamma {panel_result.corrected_persistence:.3f})"
)
print(f"Bootstrap 95%     : {low:.3f} to {high:.3f}")
print(f"Gates             : {panel_result.gate_failures or 'all passed'}")

Observations      : 108 (4 countries, 27 years)
Uncorrected Theta : 0.888 (gamma -0.084)
Corrected Theta   : 0.888 (gamma -0.072)
Bootstrap 95%     : 0.711 to 1.005
Gates             : all passed


In [5]:
# The pre-specified sensitivity: country effects only. Year effects absorb an
# additive shock common to every country in a year, and nothing more.
country_only = estimate_dynamic_panel(
    panel,
    fixed_effects="country",
    replications=299,
    bias_correction_draws=100,
)
print(f"With year effects   : {panel_result.corrected_multiplier:.3f}")
print(f"Country effects only: {country_only.corrected_multiplier:.3f}")

With year effects   : 0.888
Country effects only: 0.846


## What this notebook does not establish

The pooled estimate is not a second body of evidence. It uses the same
country-years as the country table above, and it reaches whatever precision it
has by assuming common dynamics. Where `I²` is high, that assumption is
contradicted by the same data, and the country estimates are the finding.